# 1. Environment Setup, Data Loading & Stratified Split
Welcome to the Multi-Task Learning (MTL) Model Training Pipeline! In this module, we will set up our environment by configuring our hardware acceleration and loading our pre-processed dataset.

Unlike standard classification, we will map **two** target variables simultaneously: `final_sentiment` and `primary_aspect`.

To maintain rigorous training integrity and ensure a fair comparison across all models, we perform a unified stratified split (70% Train, 15% Validation, 15% Test). We use the `stratify` parameter on our sentiment distribution to preserve the perfectly balanced 1:1:1 class ratio across all sub-datasets, while keeping both targets correctly mapped for the Transformer. Finally, we print the value counts for verification.



In [ ]:
# =====================================================
# MOUNT GOOGLE DRIVE
# =====================================================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==========================================
# MODULE 1: SETUP, LOAD & STRATIFIED SPLIT
# ==========================================
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
import torch

# 1.1 Hardware Setup
torch.cuda.empty_cache()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active device: {device}")

# 1.2 Load Data
INPUT_FOLDER = Path("/content/drive/MyDrive/Colab Notebooks/TNL/Project/PREPROCESS DATASET")
INPUT_FILE = INPUT_FOLDER / "14_final_training_data.json"

df = pd.read_json(INPUT_FILE)

# 1.3 Handle Empty/Missing Aspects Safely instead of dropping them
# If primary_aspect is missing, empty, or NaN, fill it with "General"
df["primary_aspect"] = df["primary_aspect"].fillna("General").astype(str).str.strip()
df["primary_aspect"] = df["primary_aspect"].replace("", "General")

# 1.4 Map Dual Labels (Sentiment & Aspect)
sentiment2id = {"negative": 0, "neutral": 1, "positive": 2}
df["sentiment_id"] = df["final_sentiment"].map(sentiment2id)

unique_aspects = df["primary_aspect"].unique().tolist()
aspect2id = {label: i for i, label in enumerate(unique_aspects)}
id2aspect = {i: label for i, label in enumerate(unique_aspects)}
df["aspect_id"] = df["primary_aspect"].map(aspect2id)

# 1.5 Unified 70/15/15 Split WITH Stratify (using your original proportions)
df_train, df_temp = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["final_sentiment"]
)

df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp["final_sentiment"]
)

print("--- Unified Data Splits Completed ---")
print(f"Training Set: {len(df_train)} samples")
print(f"Validation Set: {len(df_val)} samples")
print(f"Testing Set: {len(df_test)} samples")
print(f"Total Samples Kept: {len(df_train) + len(df_val) + len(df_test)}")
print(f"Tracked Aspects: {unique_aspects}")

# 2. Transformer Data Preparation
Transformers require data to be passed via structured Hugging Face Dataset objects. In this module, we construct a custom PyTorch `MultiTaskDataset` class. It utilizes the pre-trained RoBERTa tokenizer to convert raw text into token IDs and attention masks while keeping both target label IDs safely mapped.

In [ ]:
# ==========================================
# MODULE 2: PREPARE DATASETS FOR HUGGINGFACE
# ==========================================
from transformers import AutoTokenizer
from torch.utils.data import Dataset

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MultiTaskDataset(Dataset):
    def __init__(self, texts, sentiment_labels, aspect_labels):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding="max_length", max_length=128)
        self.sentiment_labels = sentiment_labels.tolist()
        self.aspect_labels = aspect_labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['sentiment_labels'] = torch.tensor(self.sentiment_labels[idx], dtype=torch.long)
        item['aspect_labels'] = torch.tensor(self.aspect_labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.sentiment_labels)

train_dataset = MultiTaskDataset(df_train["translated_sentence"], df_train["sentiment_id"], df_train["aspect_id"])
val_dataset = MultiTaskDataset(df_val["translated_sentence"], df_val["sentiment_id"], df_val["aspect_id"])
test_dataset = MultiTaskDataset(df_test["translated_sentence"], df_test["sentiment_id"], df_test["aspect_id"])

print("✅ Transformer Multi-Task Datasets are ready.")

# 3. Fine-Tune Multi-Task RoBERTa Model
We implement a deep learning architecture using a shared **RoBERTa Encoder** base linked to two distinct classification heads. To optimize this custom network, we override the Hugging Face `Trainer` class to compute a combined Joint Cross-Entropy Loss ($Loss_{Total} = Loss_{Sentiment} + Loss_{Aspect}$).

The model is configured with a high-patience Early Stopping callback to halt execution gracefully when validation loss stops improving.

In [ ]:
#  !pip install --upgrade torchvision

In [ ]:
# 1. Uninstall the conflicting default Colab packages
!pip uninstall -y torch torchvision torchaudio

# 2. Force install the perfectly matching versions
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# 3. Ensure your Hugging Face libraries are up to date
!pip install --upgrade transformers datasets evaluate scikit-learn

In [ ]:
# ==========================================
# MODULE 3: MULTI-TASK ARCHITECTURE & TRAINING
# ==========================================
import torch.nn as nn
from transformers import AutoModel, Trainer, TrainingArguments, EarlyStoppingCallback

# 3.1 Custom Architecture
class MultiTaskRoberta(nn.Module):
    def __init__(self, model_name, num_sentiments, num_aspects):
        super(MultiTaskRoberta, self).__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        hidden_size = self.roberta.config.hidden_size

        # Dual Classification Heads
        self.sentiment_classifier = nn.Linear(hidden_size, num_sentiments)
        self.aspect_classifier = nn.Linear(hidden_size, num_aspects)

    def forward(self, input_ids, attention_mask, sentiment_labels=None, aspect_labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :] # Extract [CLS] token

        sentiment_logits = self.sentiment_classifier(cls_output)
        aspect_logits = self.aspect_classifier(cls_output)

        loss = None
        if sentiment_labels is not None and aspect_labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            # Joint Loss calculation
            loss = loss_fct(sentiment_logits, sentiment_labels) + loss_fct(aspect_logits, aspect_labels)

        return {"loss": loss, "sentiment_logits": sentiment_logits, "aspect_logits": aspect_logits}

# 3.2 Custom Trainer
class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        sentiment_labels = inputs.pop("sentiment_labels")
        aspect_labels = inputs.pop("aspect_labels")
        outputs = model(**inputs, sentiment_labels=sentiment_labels, aspect_labels=aspect_labels)
        return (outputs["loss"], outputs) if return_outputs else outputs["loss"]

# 3.3 Initialize Model
model = MultiTaskRoberta(MODEL_NAME, num_sentiments=3, num_aspects=len(unique_aspects)).to(device)

# 3.4 Training Arguments
training_args = TrainingArguments(
    output_dir="/roberta_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    num_train_epochs=30,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none",
    remove_unused_columns=False # Crucial for MTL
)

trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Starting Multi-Task RoBERTa Fine-Tuning...")
trainer.train()

# 4. Model Evaluation & Metrics Export
Now that our Multi-Task RoBERTa model has finished fine-tuning, we put it into evaluation mode to generate predictions on our unseen `test_dataset`. We extract outputs from both target heads and print comprehensive classification reports for both tasks to analyze Precision, Recall, and F1-Scores.

In [ ]:
import torch
# ==========================================
# MODULE 4: EVALUATION & EXPORT
# ==========================================
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print("Evaluating Multi-Task RoBERTa on the test set...")

# 4.1 Generate Predictions Manually
model.eval()
y_pred_sent_tf, y_pred_asp_tf = [], []
y_true_sent, y_true_asp = df_test["sentiment_id"].tolist(), df_test["aspect_id"].tolist()

with torch.no_grad():
    for i in range(len(test_dataset)):
        inputs = {
            'input_ids': test_dataset[i]['input_ids'].unsqueeze(0).to(device),
            'attention_mask': test_dataset[i]['attention_mask'].unsqueeze(0).to(device)
        }
        outputs = model(**inputs)
        y_pred_sent_tf.append(torch.argmax(outputs['sentiment_logits'], dim=-1).item())
        y_pred_asp_tf.append(torch.argmax(outputs['aspect_logits'], dim=-1).item())

# 4.2 Display Sentiment Results
print("\n--- RoBERTa Multi-Task Results (SENTIMENT) ---")
print(classification_report(y_true_sent, y_pred_sent_tf, target_names=["negative", "neutral", "positive"]))

# 4.3 Display Aspect Results
print("\n--- RoBERTa Multi-Task Results (ASPECT) ---")
# Fix: Dynamically get the labels present in the test set for aspect classification
actual_aspect_labels = sorted(list(set(y_true_asp + y_pred_asp_tf)))
actual_aspect_target_names = [id2aspect[label] for label in actual_aspect_labels]
print(classification_report(y_true_asp, y_pred_asp_tf, labels=actual_aspect_labels, target_names=actual_aspect_target_names))

# 5. Results Visualization
To visually evaluate how effectively our model performs across both domains, we generate two core performance diagnostic charts:
1. **Side-by-Side Confusion Matrices**: Plots correct vs. misclassified labels for both Sentiment (Greens) and Aspect (Blues) to reveal exactly where the network gets confused.
2. **Performance Metric Comparison**: A grouped bar chart displaying the final Accuracy and Macro F1 scores achieved on each distinct task.

In [ ]:
# ==========================================
# MODULE 5: RESULT VISUALIZATION
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay
import pandas as pd

print("Generating Visualizations...")

# 5.1 Plot Side-by-Side Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Sentiment Confusion Matrix (Greens)
ConfusionMatrixDisplay.from_predictions(
    y_true_sent, y_pred_sent_tf,
    display_labels=["negative", "neutral", "positive"],
    cmap="Greens", ax=axes[0]
)
axes[0].set_title("MTL RoBERTa - Sentiment", fontsize=14, fontweight="bold")
axes[0].grid(False)

# Aspect Confusion Matrix (Blues)
ConfusionMatrixDisplay.from_predictions(
    y_true_asp, y_pred_asp_tf,
    display_labels=actual_aspect_target_names, # Fixed: Use actual_aspect_target_names
    cmap="Blues", ax=axes[1],
    xticks_rotation=45 # Rotated for better readability if aspect names are long
)
axes[1].set_title("MTL RoBERTa - Aspect", fontsize=14, fontweight="bold")
axes[1].grid(False)

plt.tight_layout()
plt.savefig("mtl_confusion_matrices.png", dpi=300)
plt.show()

# 5.2 Calculate Final Metrics for Bar Chart
results = {
    "Task": ["Sentiment Classification", "Aspect Classification"],
    "Accuracy": [
        accuracy_score(y_true_sent, y_pred_sent_tf),
        accuracy_score(y_true_asp, y_pred_asp_tf)
    ],
    "Macro F1": [
        f1_score(y_true_sent, y_pred_sent_tf, average="macro"),
        f1_score(y_true_asp, y_pred_asp_tf, average="macro")
    ]
}

# Convert to DataFrame and melt for Seaborn plotting
df_results = pd.DataFrame(results)
df_melted = df_results.melt(id_vars="Task", var_name="Metric", value_name="Score")

# 5.3 Plot Performance Comparison Bar Chart
plt.figure(figsize=(8, 6))
sns.set_theme(style="whitegrid")

# Create grouped bar chart
ax = sns.barplot(data=df_melted, x="Task", y="Score", hue="Metric", palette="viridis")

plt.title("Multi-Task RoBERTa: Sentiment vs. Aspect Performance", fontsize=16, fontweight="bold")
plt.ylim(0, 1.1)
plt.legend(loc='lower right', title="Metric")

# Annotate exact numerical values on top of the bars
for p in ax.patches:
    ax.annotate(format(p.get_height(), '.3f'),
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 10), textcoords='offset points',
                fontsize=11)

plt.tight_layout()
plt.savefig("mtl_performance_comparison.png", dpi=300)
plt.show()

# 6. Save & Download Model
Because our Multi-Task Learning architecture is a custom PyTorch class (`MultiTaskRoberta`) rather than a standard Hugging Face pipeline, we cannot rely on the default `trainer.save_model()` function.

Instead, in this final module, we save the pre-trained **Tokenizer** using the standard API and extract the model's **state_dict** (the dictionary containing all our freshly fine-tuned weights) using native PyTorch commands. Finally, we package these components into a ZIP archive and initiate a direct download to the local machine for future deployment or web app integration.

In [ ]:
# ==========================================
# MODULE 6: SAVE & DOWNLOAD MODEL
# ==========================================
import torch
import shutil
import os
from google.colab import files

print("Saving model and tokenizer...")

# 1. Define where to save the files locally in Colab
save_path = "/content/my_mtl_model"
os.makedirs(save_path, exist_ok=True)

# 2. Save the Tokenizer (so you can process new text later)
tokenizer.save_pretrained(save_path)

# 3. Save the Custom Model Weights
# We save the 'state_dict' because this is a custom multi-head architecture
torch.save(model.state_dict(), f"{save_path}/mtl_roberta_weights.pt")

print("Files saved! Zipping the folder...")

# 4. Zip the folder using shutil
shutil.make_archive("my_mtl_model", 'zip', save_path)

# 5. Trigger the download to your local machine
print("Initiating download...")
files.download("my_mtl_model.zip")